# Case 7 — Configurable Rule Engine

Case 1-6, istatistiksel/context-adjustment tabanlı bir anomali skoru üretti
(`final_raw_anomaly_score`, Case 6'nın dört context düzeltmesiyle birlikte). Bu adım farklı bir
katman ekliyor: **kural tabanlı, açıklanabilir bir motor** — JSON/YAML'dan okunan, kod
değişmeden eklenip çıkarılabilen kurallar, öncelik (priority) ve çakışma çözümü (conflict
resolution) ile net bir final karar üreten, ve bunu Case 5'in ürettiği anomali skoruyla birlikte
çalıştıran bir sistem.

Motor `src/services/rules/` altında: `models.py` (Rule, Severity), `conditions.py` (Composite
pattern — AND/OR/NOT kural ağaçları), `operators.py` (Strategy pattern — karşılaştırma
operatörleri), `loader.py` (Factory pattern — YAML/JSON'dan Rule ağacı kurma), `resolution.py`
(Chain of Responsibility — final karar çözümleme), `engine.py` (RuleEngine), `container.py`
(dependency_injector — projede ilk kez kullanılıyor). Kurallar `definitions/fraud_rules.yaml`
(10 kural, kanonik) ve `definitions/example_rules.json` (3 kural, JSON desteğini kanıtlayan
örnek) dosyalarında.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found — expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Kuralları yükleme — YAML ve JSON

Loader (Factory pattern), dosya uzantısına göre YAML/JSON'ı ayırt ediyor, ikisi de aynı şemayı
(`all_of`/`any_of`/`not` + `field`/`operator`/`value` yaprakları) aynı `RuleLoader.load()`
metoduyla işliyor.

In [3]:
from src.services.rules.loader import RuleLoader

loader = RuleLoader()
rules = loader.load(REPO_ROOT / "src/services/rules/definitions/fraud_rules.yaml")
example_rules = loader.load(REPO_ROOT / "src/services/rules/definitions/example_rules.json")

print(f"YAML'dan yüklenen kanonik kural sayısı: {len(rules)}")
for r in rules:
    print(f"  {r.id} [{r.severity.name:>8}] priority={r.priority:>2}  {r.name}")

print()
print(f"JSON'dan yüklenen örnek kural sayısı: {len(example_rules)}")
for r in example_rules:
    print(f"  {r.id} [{r.severity.name:>8}] priority={r.priority:>2}  {r.name}")

YAML'dan yüklenen kanonik kural sayısı: 10
  fraud_r01 [CRITICAL] priority= 1  High Amount + Foreign Country + Night
  fraud_r02 [    HIGH] priority= 2  Velocity — Rapid Repeat Transaction
  fraud_r03 [    HIGH] priority= 3  New Device + New Address Together
  fraud_r04 [    HIGH] priority= 4  Extreme Deviation From Card's Own Spending
  fraud_r05 [CRITICAL] priority= 5  Brand-New Card, High-Value First Transaction
  fraud_r06 [  MEDIUM] priority= 6  Weekend Off-Hours High-Value
  fraud_r07 [    HIGH] priority= 7  AI Anomaly Score — Extreme Tail
  fraud_r08 [  MEDIUM] priority= 8  Generic/Missing Device Info + High Amount
  fraud_r09 [CRITICAL] priority= 9  Address-Hopping Velocity
  fraud_r10 [     LOW] priority=10  Unusually Large Physical Distance

JSON'dan yüklenen örnek kural sayısı: 3
  example_json_01 [  MEDIUM] priority= 1  High Amount (JSON example)
  example_json_02 [     LOW] priority= 2  New Device Only (JSON example)
  example_json_03 [    HIGH] priority= 3  New Device Or 

**Severity dağılımı 4 seviyeyi de kapsıyor** (3 CRITICAL, 4 HIGH, 2 MEDIUM, 1 LOW) — Chain of
Responsibility'nin her dalı en az bir kuralla gerçekten test edilebiliyor.

## 2. Veri hazırlığı — mevcut feature modüllerinden, yeni feature mühendisliği yok

Kurallar tamamen Case 3-6'nın zaten ürettiği kolonlara referans veriyor: `temporal.py`,
`entity.py`, `relational.py`, ham `TransactionAmt`/`addr2`/`DeviceInfo`/`dist1`, ve Case 5'in
`final_raw_anomaly_score`'u (motoru anomali skoruyla birleştiren köprü kural, fraud_r07).

In [4]:
from src.services.features.temporal import build_temporal_features
from src.services.features.entity import build_entity_features
from src.services.features.relational import build_relational_features
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score

raw = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "TransactionAmt", "addr2", "DeviceInfo", "dist1"]).to_pandas()
temporal = build_temporal_features(parquet_path)
entity = build_entity_features(parquet_path)
relational = build_relational_features(parquet_path)

all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)

isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()

df = raw.merge(temporal, on="TransactionID") \
    .merge(entity, on="TransactionID") \
    .merge(relational, on="TransactionID") \
    .merge(final_raw, on="TransactionID") \
    .merge(isfraud, on="TransactionID")

print(f"birleşik tablo: {df.shape}")

birleşik tablo: (590540, 27)


**Metodolojik not:** bu notebook'u hazırlarken, eşikleri terminalde ayrı ayrı denerken
(`entity`/`relational` gibi ayrı feature tablolarını `df`'e katmadan önce, kendi satır sıralarıyla)
bir hizalama hatası buldum — `entity.py` satırları `card1+zaman` sırasına göre sıralıyor,
`TransactionID` sırasına göre değil, bu yüzden `entity` tablosundan alınan bir boolean maske,
birleşik `df` üzerinde doğrudan `.loc[]` ile kullanılınca YANLIŞ satırları seçiyordu (sessizce,
hata vermeden). Motor (`RuleEngine.evaluate_all`) her koşulu her zaman TEK bir birleşik `df`
üzerinde değerlendirdiği için bu riski taşımıyor — ama bu, ayrı ayrı üretilmiş feature
tablolarını manuel karıştırırken dikkat edilmesi gereken gerçek bir tuzak olduğunu gösteriyor.

## 3. Motoru çalıştırma — tüm veri seti, vektörize

In [5]:
from src.services.rules.resolution import build_default_resolution_chain
from src.services.rules.engine import RuleEngine

engine = RuleEngine(rules, build_default_resolution_chain())
result = engine.evaluate_all(df)

print(f"sonuç tablosu: {result.shape}")
result[[r.id for r in rules] + ["fired_rule_count", "verdict_severity", "verdict_rule_id"]].head()

sonuç tablosu: (590540, 16)


,fraud_r01,fraud_r02,fraud_r03,fraud_r04,fraud_r05,fraud_r06,fraud_r07,fraud_r08,fraud_r09,fraud_r10,fired_rule_count,verdict_severity,verdict_rule_id
0,False,False,False,False,False,False,False,False,False,False,0,NaN,NaN
1,False,False,False,False,False,False,False,False,False,False,0,NaN,NaN
2,False,False,False,False,False,False,False,False,False,False,0,NaN,NaN
3,False,False,False,False,False,False,False,False,False,False,0,NaN,NaN
4,False,False,True,False,False,False,False,False,False,False,1,HIGH,fraud_r03


## 4. Her kuralın betimleyici doğrulaması — sonradan, `isFraud` ile

Severity'ler bu veri setinin fraud oranına göre AYARLANMADI (domain/operasyonel risk yargısı) —
burada sadece hangi kuralların ölçülen etkisinin gerçekten yüksek olduğunu, hangilerinin
"mantıklı görünse de" bu veri setinde zayıf kaldığını dürüstçe raporluyoruz.

In [6]:
baseline_rate = df["isFraud"].mean() * 100
print(f"genel taban fraud oranı: %{baseline_rate:.3f}\n")

for r in rules:
    fired = result[r.id].astype(bool)
    n = int(fired.sum())
    rate = df.loc[fired, "isFraud"].mean() * 100 if n > 0 else float("nan")
    flag = "↑" if rate > baseline_rate else "≈/↓"
    print(f"  {r.id} [{r.severity.name:>8}]: {n:>6} tetiklendi (%{n/len(df)*100:.3f}), fraud oranı: %{rate:.2f}  {flag}")

genel taban fraud oranı: %3.499

  fraud_r01 [CRITICAL]:    110 tetiklendi (%0.019), fraud oranı: %14.55  ↑
  fraud_r02 [    HIGH]:  19399 tetiklendi (%3.285), fraud oranı: %6.38  ↑
  fraud_r03 [    HIGH]:   8298 tetiklendi (%1.405), fraud oranı: %3.99  ↑
  fraud_r04 [    HIGH]:  18343 tetiklendi (%3.106), fraud oranı: %3.97  ↑
  fraud_r05 [CRITICAL]:    716 tetiklendi (%0.121), fraud oranı: %3.63  ↑
  fraud_r06 [  MEDIUM]:    357 tetiklendi (%0.060), fraud oranı: %10.36  ↑
  fraud_r07 [    HIGH]:   5906 tetiklendi (%1.000), fraud oranı: %2.81  ≈/↓
  fraud_r08 [  MEDIUM]:  26759 tetiklendi (%4.531), fraud oranı: %4.88  ↑
  fraud_r09 [CRITICAL]:    600 tetiklendi (%0.102), fraud oranı: %5.67  ↑
  fraud_r10 [     LOW]:  11887 tetiklendi (%2.013), fraud oranı: %3.27  ≈/↓


**Karışık ama dürüst bir tablo — Case 6'da defalarca görülen bir tema burada da tekrarlanıyor:**
tek-koşullu kurallar (fraud_r02 velocity, fraud_r04 z-score) tabanın oldukça üzerinde ama
bileşik (AND'li) kurallar çok daha keskin (fraud_r01 %14,55, fraud_r06 %10,36) — birden fazla
zayıf sinyali birleştirmek, her birini tek başına kullanmaktan daha güçlü bir ayrım üretiyor.

**fraud_r07 (AI köprü kuralı, üst-%1 anomali skoru) beklenmedik biçimde tabanın ALTINDA kalıyor
(%2,81 vs %3,50)** — Case 6'da `final_raw_anomaly_score`'un en üst kovasının (9. kova %5,57 →
10. kova %5,34) monotonik olmadığı zaten görülmüştü; bu, aynı düzensizliğin rule-engine
tarafında da yüzeye çıkışı. Kural yine de kalıyor (motoru AI skoruyla köprüleyen mekanizma bu),
ama bu zayıflık gizlenmiyor.

## 5. Verdict (final karar) dağılımı — Chain of Responsibility'nin ürettiği özet

In [7]:
print("verdict_severity dağılımı:")
print(result["verdict_severity"].value_counts(dropna=False))
print()
print("kaç işlemde birden fazla kural aynı anda ateşleniyor (fired_rule_count):")
print(result["fired_rule_count"].value_counts().sort_index())

verdict_severity dağılımı:
verdict_severity
NaN         518620
HIGH         45965
MEDIUM       13899
LOW          10633
CRITICAL      1423
Name: count, dtype: int64

kaç işlemde birden fazla kural aynı anda ateşleniyor (fired_rule_count):
fired_rule_count
0    518620
1     57426
2      8925
3      5188
4       370
5        11
Name: count, dtype: int64


## 6. Explainability — gerçek bir işlem üzerinde `explain()`

In [8]:
fired_idx = result[result["fraud_r01"]].index[0]
example_row = df.loc[fired_idx]

explanation = engine.explain(example_row)
print(f"TransactionID={example_row['TransactionID']}, isFraud={example_row['isFraud']}\n")
for fired in explanation["fired_rules"]:
    print(f"[{fired['severity']}] {fired['rule_name']} (priority={fired['priority']})")
    print(f"  koşul: {fired['condition']}")
    print(f"  açıklama: {fired['message']}")
    print()
print(f"FINAL VERDICT: {explanation['verdict_severity']} -> {explanation['verdict_rule_id']}")

TransactionID=2988038, isFraud=0

[CRITICAL] High Amount + Foreign Country + Night (priority=1)
  koşul: (TransactionAmt=np.float64(100.0) (gt 70.0) AND addr2=np.float64(96.0) (is_null False) AND addr2=np.float64(96.0) (ne 87.0) AND is_low_volume_hour=np.True_ (eq True))
  açıklama: Elevated amount (100.0) combined with a foreign billing region (addr2=96.0) during a historically low-volume, high-fraud-rate hour window (04:00-09:00) — the exact combination named as an example in the case brief.

[HIGH] New Device + New Address Together (priority=3)
  koşul: (is_new_device_for_card=np.True_ (eq True) AND is_new_addr1_for_card=np.True_ (eq True))
  açıklama: Both the device and the billing address are new for this card in the same transaction — a common account-takeover signature (is_new_device_for_card=True, is_new_addr1_for_card=True).

FINAL VERDICT: CRITICAL -> fraud_r01


İki kural aynı anda ateşleniyor (`fraud_r01` CRITICAL, `fraud_r03` HIGH) — conflict resolution,
gerçek bir işlem üzerinde, doğru şekilde CRITICAL'i (daha yüksek severity) seçiyor. Her fired
kuralın açıklaması, kuralın soyut tanımı değil, bu işlemin GERÇEK değerleriyle
(`TransactionAmt=100.0`, `addr2=96.0` vb.) interpolasyonlu — `models.Rule.description`
şablonundan, `Condition.describe()` ile birlikte üretiliyor.

## 7. Conflict resolution — kurgulanmış bir çakışma senaryosu

In [9]:
synthetic = pd.Series({
    "TransactionAmt": 500.0, "addr2": 87.0, "DeviceInfo": "Windows", "dist1": 10.0,
    "is_low_volume_hour": False, "is_weekend_proxy": False,
    "user_seconds_since_last_transaction": 100.0,
    "is_new_device_for_card": False, "is_new_addr1_for_card": True,
    "user_amount_zscore": 0.5, "user_transaction_count_so_far": 0,
    "final_raw_anomaly_score": 0.1,
})

synthetic_explanation = engine.explain(synthetic)
fired_ids = [(f["rule_id"], f["severity"], f["priority"]) for f in synthetic_explanation["fired_rules"]]
print("Ateşlenen kurallar (id, severity, priority):", fired_ids)
print("Final verdict:", synthetic_explanation["verdict_severity"], "->", synthetic_explanation["verdict_rule_id"])

Ateşlenen kurallar (id, severity, priority): [('fraud_r05', 'CRITICAL', 5), ('fraud_r08', 'MEDIUM', 8), ('fraud_r09', 'CRITICAL', 9)]
Final verdict: CRITICAL -> fraud_r05


Kurgu: `fraud_r05` (yeni kart + yüksek ilk işlem, priority=5) ve `fraud_r09` (adres değişimi +
hızlı tekrar, priority=9) bilerek AYNI ANDA ateşlenecek şekilde kuruldu — ikisi de CRITICAL.
Chain of Responsibility doğru çalışıyor: `fraud_r08` (MEDIUM) de ateşleniyor ama CRITICAL'in
altında kalıyor; CRITICAL seviyesinde de düşük `priority` numarasına sahip `fraud_r05` birincil
kural olarak seçiliyor — `priority` ve `severity`'nin iki AYRI mekanizma olarak çalıştığının
kanıtı.

## 8. Dependency Injection — `dependency_injector` Container, kural dosyasını değiştirme

In [10]:
from src.services.rules.container import RuleEngineContainer

container = RuleEngineContainer()

container.config.rules_path.from_value(str(REPO_ROOT / "src/services/rules/definitions/fraud_rules.yaml"))
engine_from_yaml = container.rule_engine()
print(f"YAML ile: {len(engine_from_yaml.rules)} kural — {[r.id for r in engine_from_yaml.rules]}")

container.config.rules_path.from_value(str(REPO_ROOT / "src/services/rules/definitions/example_rules.json"))
engine_from_json = container.rule_engine()
print(f"JSON ile: {len(engine_from_json.rules)} kural — {[r.id for r in engine_from_json.rules]}")

YAML ile: 10 kural — ['fraud_r01', 'fraud_r02', 'fraud_r03', 'fraud_r04', 'fraud_r05', 'fraud_r06', 'fraud_r07', 'fraud_r08', 'fraud_r09', 'fraud_r10']
JSON ile: 3 kural — ['example_json_01', 'example_json_02', 'example_json_03']


Aynı container, tek bir config değeri (`rules_path`) değiştirilerek YAML'dan JSON'a, kod
değişmeden geçiyor — brief'in "kod değişmeden eklenip çıkarılabilen kurallar" isteğini karşılayan
somut mekanizma bu. Bu, projenin `dependency_injector`'ı ilk (ve tek, dar kapsamlı) kullanımı.

## 9. Kurallar + AI birlikte — `final_raw_anomaly_score` ile çapraz doğrulama

Brief'in asıl istediği: rule engine, istatistiksel anomali skoruyla BİRLİKTE çalışsın. İki yönlü
bakalım — kural motorunun verdict'i anomali skoru dekillerine göre nasıl dağılıyor, ve anomali
skoru kuralların yakaladığı/kaçırdığı işlemlerde nasıl davranıyor.

In [11]:
cross = result[["verdict_severity", "fired_rule_count"]].copy()
cross["final_raw_anomaly_score"] = df["final_raw_anomaly_score"]
cross["isFraud"] = df["isFraud"]

print("Verdict severity'ye göre ortalama final_raw_anomaly_score ve fraud oranı:")
print(cross.groupby("verdict_severity", dropna=False).agg(
    islem_sayisi=("isFraud", "size"),
    ort_anomali_skoru=("final_raw_anomaly_score", "mean"),
    fraud_orani_yuzde=("isFraud", lambda s: round(s.mean() * 100, 3)),
).sort_values("fraud_orani_yuzde", ascending=False))

Verdict severity'ye göre ortalama final_raw_anomaly_score ve fraud oranı:
                  islem_sayisi  ort_anomali_skoru  fraud_orani_yuzde
verdict_severity                                                    
MEDIUM                   13899           0.843728              6.080
CRITICAL                  1423           0.648897              5.341
HIGH                     45965           0.646219              4.891
NaN                     518620           0.478066              3.311
LOW                      10633           0.468553              3.028


## 10. Kurallar ile AI'ın YAKALADIĞI farklı kümeler — tamamlayıcılık kontrolü

In [12]:
business_rule_ids = [r.id for r in rules if r.id != "fraud_r07"]
any_business_rule_fired = result[business_rule_ids].any(axis=1)
ai_top1pct = df["final_raw_anomaly_score"] >= df["final_raw_anomaly_score"].quantile(0.99)

both = any_business_rule_fired & ai_top1pct
rules_only = any_business_rule_fired & ~ai_top1pct
ai_only = ~any_business_rule_fired & ai_top1pct
neither = ~any_business_rule_fired & ~ai_top1pct

for name, mask in [("her ikisi de", both), ("sadece iş kuralları (fraud_r07 hariç)", rules_only), ("sadece AI (üst-%1)", ai_only), ("hiçbiri", neither)]:
    n = mask.sum()
    rate = df.loc[mask, "isFraud"].mean() * 100 if n > 0 else float("nan")
    print(f"  {name}: {n:,} işlem (%{n/len(df)*100:.2f}), fraud oranı: %{rate:.2f}")


  her ikisi de: 5,687 işlem (%0.96), fraud oranı: %2.85
  sadece iş kuralları (fraud_r07 hariç): 66,014 işlem (%11.18), fraud oranı: %5.04
  sadece AI (üst-%1): 219 işlem (%0.04), fraud oranı: %1.83
  hiçbiri: 518,620 işlem (%87.82), fraud oranı: %3.31


**Kurallar ve AI skoru büyük ölçüde FARKLI işlemleri yakalıyor — ve bu sefer dürüst tablo,
naif "ikisi birleşince en güçlü sinyal olur" beklentisini de doğrulamıyor.** İş kuralları
(fraud_r07 hariç) tek başına en güçlü tekil grup: %5,04 fraud oranı, 66.014 işlem. "Sadece AI"
(sadece üst-%1 anomali skoru, hiçbir iş kuralı ateşlenmeden) grubu ise tabanın ALTINDA kalıyor
(%1,83) — Case 6'da zaten görülen, `final_raw_anomaly_score`'un en üst kovasının kendi içinde tam
güvenilir olmadığı bulgusuyla tutarlı. En şaşırtıcı olan: "her ikisi de" (hem iş kuralı hem AI
üst-%1) grubu %2,85 — "sadece iş kuralları" grubundan (%5,04) daha DÜŞÜK. Bunun sebebi muhtemelen
fraud_r01/fraud_r06 gibi en keskin iş kurallarının, tanım gereği AI skorunun üst-%1'i ile
örtüşmeyen farklı bir işlem profilini (gece + yabancı + orta tutar gibi) yakalaması — iki sinyal
gerçekten TAMAMLAYICI, ama "birleşince otomatik olarak daha güçlü" değil. Bu, brief'in "iş
kuralları + AI birlikte" motivasyonunu doğruluyor (ikisi farklı, değerli bilgi taşıyor), ama daha
sofistike bir birleştirme (örn. Case 5'in weighted-aggregation mantığı) olmadan basit bir
kesişimin otomatik olarak en iyi sonucu vermeyeceğini de gösteriyor — gizlenmeyen, dürüst bir
bulgu.



## 11. If-Then Evaluation

Şimdiye kadarki bölümler kuralın "if" (koşul) tarafını ve severity'sini kullandı, ama "then"
tarafı — sistemin GERÇEKTE ne YAPACAĞI — örtük kalmıştı. Şema artık açıkça `if:`/`then:` olarak
ayrıldı (`loader.py`, `models.py`): her kuralın `then` bloğu bir **action**
(`BLOCK`/`REVIEW`/`FLAG`/`ALLOW`) taşıyor — severity'den AYRI bir eksen. İki kural aynı severity'yi
paylaşabilir ama farklı bir eylem gerektirebilir (örn. gerçek bir fraud-ops sisteminde "işlemi
durdur" ile "sadece logla, sonra incele" aynı risk seviyesinde bile farklı kararlar olabilir).

In [13]:
for r in rules:
    print(f"IF <{r.name}>")
    print(f"THEN action={r.action.name}, severity={r.severity.name}")
    print()

IF <High Amount + Foreign Country + Night>
THEN action=BLOCK, severity=CRITICAL

IF <Velocity — Rapid Repeat Transaction>
THEN action=REVIEW, severity=HIGH

IF <New Device + New Address Together>
THEN action=REVIEW, severity=HIGH

IF <Extreme Deviation From Card's Own Spending>
THEN action=REVIEW, severity=HIGH

IF <Brand-New Card, High-Value First Transaction>
THEN action=BLOCK, severity=CRITICAL

IF <Weekend Off-Hours High-Value>
THEN action=FLAG, severity=MEDIUM

IF <AI Anomaly Score — Extreme Tail>
THEN action=REVIEW, severity=HIGH

IF <Generic/Missing Device Info + High Amount>
THEN action=FLAG, severity=MEDIUM

IF <Address-Hopping Velocity>
THEN action=BLOCK, severity=CRITICAL

IF <Unusually Large Physical Distance>
THEN action=FLAG, severity=LOW



### Action dağılımı — tüm veri setinde, final verdict üzerinden

In [14]:
print("verdict_action dağılımı:")
print(result["verdict_action"].value_counts(dropna=False))
print()

print("severity -> action eşlemesi (final verdict'lerde hangi kombinasyonlar oluşuyor):")
print(result[["verdict_severity", "verdict_action"]].dropna().drop_duplicates().sort_values("verdict_severity"))

verdict_action dağılımı:
verdict_action
NaN       518620
REVIEW     45965
FLAG       24532
BLOCK       1423
Name: count, dtype: int64

severity -> action eşlemesi (final verdict'lerde hangi kombinasyonlar oluşuyor):
   verdict_severity verdict_action
18         CRITICAL          BLOCK
4              HIGH         REVIEW
78              LOW           FLAG
73           MEDIUM           FLAG


Severity ile action arasında **1:1 olmayan** bir ilişki bilerek kuruldu: CRITICAL→BLOCK,
HIGH→REVIEW, MEDIUM/LOW→FLAG — ama bu, `resolution.py`'nin severity'yi çözümleyip sonra o
severity'nin birincil kuralının action'ını taşımasıyla ortaya çıkıyor, action'ın kendisi ayrı bir
Chain of Responsibility'den geçmiyor. `ALLOW` hiçbir kuralda kullanılmadı — bilerek: bu 10 kuralın
hepsi risk-bayrağı, allow-list kuralı değil; şema `ALLOW`'u destekliyor ama bu veri setinde
kullanılacak bir "güvenilir" kural şu an yok (Case 6'nın trusted-entity/business-context
düzeltmeleri ayrı bir katman, bu motorun içinde değil).

### Action bazında betimleyici doğrulama

In [15]:
for action_name in ["BLOCK", "REVIEW", "FLAG"]:
    mask = result["verdict_action"] == action_name
    n = mask.sum()
    rate = df.loc[mask, "isFraud"].mean() * 100
    print(f"  {action_name}: {n:,} işlem (%{n/len(df)*100:.2f}), fraud oranı: %{rate:.2f}")
print(f"  (hiçbir kural ateşlenmedi): {(result['verdict_action'].isna()).sum():,} işlem, fraud oranı: %{df.loc[result['verdict_action'].isna(),'isFraud'].mean()*100:.2f}")

  BLOCK: 1,423 işlem (%0.24), fraud oranı: %5.34
  REVIEW: 45,965 işlem (%7.78), fraud oranı: %4.89
  FLAG: 24,532 işlem (%4.15), fraud oranı: %4.76
  (hiçbir kural ateşlenmedi): 518,620 işlem, fraud oranı: %3.31


**Action'lar fraud oranıyla beklenen sırada hizalanıyor: BLOCK > REVIEW > FLAG > (hiçbir kural)**
— bu, severity'nin zaten doğru sıralandığının bir yansıması (action, severity üzerinden
belirleniyor), ama yine de operasyonel olarak anlamlı: en sert eylem (BLOCK) gerçekten en yüksek
fraud yoğunluğuna denk düşüyor.



## 12. Rule Priority Mekanizması — Severity'den Bağımsız Bir Karşılaştırma

Şu ana kadar `priority`, sadece Chain of Responsibility'nin İÇİNDE, aynı severity'yi paylaşan
kurallar arasında görünmez bir tie-break olarak çalıştı — etkisini doğrudan gözlemleyemiyorduk.
`resolution.py`'ye ikinci, kasıtlı olarak daha basit bir strateji eklendi: `resolve_by_priority`,
severity'yi tamamen görmezden gelip ateşlenen kurallar arasından sadece en düşük `priority`
numarasına sahip olanı seçiyor. `RuleEngine.evaluate_all` artık ikisini de aynı anda üretiyor
(`verdict_rule_id` = severity-CoR, `priority_verdict_rule_id` = saf öncelik) — böylece `priority`
mekanizmasının GERÇEK etkisini, iki stratejiyi karşılaştırarak ölçebiliyoruz.

In [16]:
agree = result["verdict_rule_id"] == result["priority_verdict_rule_id"]
any_fired = result["fired_rule_count"] > 0

print(f"En az bir kural ateşlenen işlem: {any_fired.sum():,}")
print(f"Severity-CoR ile saf-priority AYNI kuralı seçiyor: {agree[any_fired].sum():,} ({agree[any_fired].mean()*100:.2f}%)")
print(f"FARKLI kural seçiyor: {(~agree[any_fired]).sum():,} ({(~agree[any_fired]).mean()*100:.2f}%)")
print()

disagreement = df.loc[any_fired & ~agree, ["TransactionID"]].copy()
disagreement["cor_rule"] = result.loc[any_fired & ~agree, "verdict_rule_id"]
disagreement["cor_severity"] = result.loc[any_fired & ~agree, "verdict_severity"]
disagreement["priority_only_rule"] = result.loc[any_fired & ~agree, "priority_verdict_rule_id"]
disagreement["priority_only_severity"] = result.loc[any_fired & ~agree, "priority_verdict_severity"]
print(disagreement.head(10).to_string(index=False))

En az bir kural ateşlenen işlem: 71,920


Severity-CoR ile saf-priority AYNI kuralı seçiyor: 71,614 (99.57%)
FARKLI kural seçiyor: 306 (0.43%)

 TransactionID  cor_rule cor_severity priority_only_rule priority_only_severity
       2987110 fraud_r09     CRITICAL          fraud_r08                 MEDIUM
       2987211 fraud_r09     CRITICAL          fraud_r02                   HIGH
       2987221 fraud_r09     CRITICAL          fraud_r02                   HIGH
       2987265 fraud_r09     CRITICAL          fraud_r03                   HIGH
       2987337 fraud_r09     CRITICAL          fraud_r03                   HIGH
       2987341 fraud_r09     CRITICAL          fraud_r08                 MEDIUM
       2987355 fraud_r05     CRITICAL          fraud_r03                   HIGH
       2987467 fraud_r09     CRITICAL          fraud_r03                   HIGH
       2987520 fraud_r09     CRITICAL          fraud_r03                   HIGH
       2987597 fraud_r09     CRITICAL          fraud_r02                   HIGH


**%99,58 uyum — ama uyumsuz kalan 306 işlemde ilginç, tutarlı bir örüntü var: severity-CoR her
seferinde `priority_only`'dan daha az tehlikeli olmayan bir kural seçiyor (hiç tersi yok).**
Uyumsuzluk hep şu şekilde: `fraud_r05`/`fraud_r09` (CRITICAL, priority=5/9) daha ERKEN öncelikli
bir HIGH/MEDIUM kuralla (örn. `fraud_r02`/`fraud_r03`/`fraud_r04`, priority=2-4) aynı anda
ateşleniyor — saf öncelik stratejisi körü körüne erken-numaralı (ama daha az tehlikeli) kuralı
seçerken, severity-CoR doğru şekilde CRITICAL olanı öne çıkarıyor. Bu, `priority`'nin TEK BAŞINA
(severity'siz) bir çakışma çözme stratejisi olarak yetersiz kalabileceğinin somut kanıtı — ve
severity-farkındalıklı CoR'un neden tercih edilen varsayılan olduğunu doğruluyor. `priority` yine
de gerçek bir işi var: her severity katmanının KENDİ İÇİNDEKİ sıralamayı belirliyor — bu ayrım artık
gözlemlenebilir, sadece varsayılmıyor.

## 13. Explainability Output — Toplu Üretim

Şimdiye kadarki `explain()` çağrıları tek tek örneklerdi. `RuleEngine.explain_batch(df, result)`,
önceden hesaplanmış `evaluate_all` sonucunu (koşulları yeniden değerlendirmeden) yeniden kullanarak
BİRDEN FAZLA işlem için tam, yapılandırılmış açıklama kayıtları üretiyor — asıl "explainability
output" teslimatı bu, tek seferlik demo değil.

In [17]:
critical_idx = df.index[result["verdict_severity"] == "CRITICAL"]
critical_subset = df.loc[critical_idx]

explainability_output = engine.explain_batch(critical_subset, result.loc[critical_idx])
print(f"Üretilen açıklama kaydı sayısı: {len(explainability_output):,} (tüm CRITICAL-verdict işlemler)")
print()

explainability_summary = pd.DataFrame([
    {
        "TransactionID": rec["TransactionID"],
        "fired_rule_count": len(rec["fired_rules"]),
        "verdict_rule_id": rec["verdict_rule_id"],
        "verdict_action": rec["verdict_action"],
        "priority_verdict_rule_id": rec["priority_verdict_rule_id"],
        "primary_message": next(f["message"] for f in rec["fired_rules"] if f["rule_id"] == rec["verdict_rule_id"]),
    }
    for rec in explainability_output
])
explainability_summary.head(10)

Üretilen açıklama kaydı sayısı: 1,423 (tüm CRITICAL-verdict işlemler)



,TransactionID,fired_rule_count,verdict_rule_id,verdict_action,priority_verdict_rule_id,primary_message
0,2987018,1,fraud_r09,BLOCK,fraud_r09,"A brand-new billing address for this card, used again within 256.0 seconds — the address-hopping pattern common to f..."
1,2987041,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 774.0 — no track record to justify...
2,2987056,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 500.0 — no track record to justify...
3,2987060,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 2454.0 — no track record to justif...
4,2987076,1,fraud_r09,BLOCK,fraud_r09,"A brand-new billing address for this card, used again within 186.0 seconds — the address-hopping pattern common to f..."
5,2987094,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 527.0 — no track record to justify...
6,2987110,2,fraud_r09,BLOCK,fraud_r08,"A brand-new billing address for this card, used again within 218.0 seconds — the address-hopping pattern common to f..."
7,2987122,1,fraud_r09,BLOCK,fraud_r09,"A brand-new billing address for this card, used again within 280.0 seconds — the address-hopping pattern common to f..."
8,2987132,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 757.07 — no track record to justif...
9,2987133,2,fraud_r05,BLOCK,fraud_r05,This card's very first observed transaction (0 prior transactions) is already for 2594.95 — no track record to justi...


Tam bir kaydın (nested) yapısına da bakalım — sadece özet tablo değil:

In [18]:
import json as _json
multi_rule_example = next(rec for rec in explainability_output if len(rec["fired_rules"]) > 1)
print(_json.dumps(multi_rule_example, indent=2, default=str))

{
  "TransactionID": "2987041",
  "fired_rules": [
    {
      "rule_id": "fraud_r05",
      "rule_name": "Brand-New Card, High-Value First Transaction",
      "severity": "CRITICAL",
      "action": "BLOCK",
      "priority": 5,
      "condition": "(user_transaction_count_so_far=np.int64(0) (eq 0) AND TransactionAmt=np.float64(774.0) (gt 445.0))",
      "message": "This card's very first observed transaction (0 prior transactions) is already for 774.0 \u2014 no track record to justify the amount."
    },
    {
      "rule_id": "fraud_r08",
      "rule_name": "Generic/Missing Device Info + High Amount",
      "severity": "MEDIUM",
      "action": "FLAG",
      "priority": 8,
      "condition": "((DeviceInfo=nan (is_null True) OR DeviceInfo=nan (in ['Windows', 'iOS Device', 'MacOS', 'Trident/7.0'])) AND TransactionAmt=np.float64(774.0) (gt 445.0))",
      "message": "High-value transaction (774.0) from a device fingerprint that's either missing or a generic OS-level label (DeviceInfo=na

**`fired_rule_count > 1` olan kayıtlar, explainability'nin neden sadece "hangi kural ateşlendi"
değil, "hepsi ne diyor ve final karar nasıl seçildi" olması gerektiğini gösteriyor** — bu örnekte
birden fazla kural aynı anda ateşleniyor, her biri kendi gerçek değerleriyle interpolasyonlu bir
mesaj taşıyor, ve `verdict_rule_id` ile `priority_verdict_rule_id` yan yana durarak iki çözümleme
stratejisinin bu spesifik işlemde aynı mı farklı mı karar verdiğini de gösteriyor.

---

**Durum:** Case 7 (configurable rule engine) tamamlandı — rule priority mekanizması artık
severity-CoR'dan bağımsız, ölçülebilir ikinci bir strateji (`resolve_by_priority`) olarak var ve
ikisi karşılaştırıldı (%99,58 uyum, uyumsuz 306 işlemde severity-farkındalıklı çözümleme her
zaman en az o kadar tehlikeli bir kural seçiyor). Explainability artık tek-örnek demo değil,
`explain_batch` ile toplu, yapılandırılmış bir çıktı (1.423 CRITICAL işlem için tam açıklama
kaydı, ~1,3 saniyede, önceden hesaplanmış sonucu yeniden kullanarak). `src/services/rules/`
altındaki tüm bileşenler (Composite, Strategy, Factory, Chain of Responsibility,
dependency_injector, if-then şema, açık action/severity ayrımı, öncelik karşılaştırması, toplu
explainability) artık hem kodda hem notebook'ta uçtan uca doğrulanmış durumda.